# Cadenas de ganancias: productos en coordenadas logarítmicas

Una aplicación de Alpha Calculus, con pasos visibles, código y límites.

Este cuaderno es autónomo. Sólo necesitas logaritmos y Python básico.
Al terminar podrás explicar contribuciones multiplicativas, actualizar
una etapa conocida y reconocer límites de rango y precisión.

1. Elegir coordenadas para una composición multiplicativa.
2. Sustituir una etapa y contrastar tres rutas de cálculo.
3. Contar el trabajo de consultas repetidas.
4. Comprobar rango extremo y cancelación histórica.

Ejecuta todas las celdas en orden desde un runtime nuevo. Sólo usamos
la biblioteca estándar: no se necesitan otro cuaderno, el libro, red,
datos privados ni acceso a Drive. Las tablas se exportan a
`resultados/cadenas_de_ganancias/` como CSV.

In [1]:
import csv
import json
import math
import platform
import random
from fractions import Fraction
from pathlib import Path

RESULTADOS = Path("resultados") / "cadenas_de_ganancias"
RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("Precisión de float: aproximadamente 16 cifras significativas.")

Python: 3.14.3
Precisión de float: aproximadamente 16 cifras significativas.


## 1. Reconocer una estructura útil

Un generador $α$ y su inversa permiten expresar operaciones en otras
coordenadas. Con $α(t)=e^t$, de $ℝ$ a $(0,∞)$, la suma transportada es

$$x\oplus_\alpha y
=\exp(\log x+\log y)=xy.$$

El neutro aditivo Alpha es **1**, porque $\log1=0$. Una atenuación
$0<x<1$ tiene coordenada negativa. No admitimos cero ni valores negativos
en esta elección de generador.

La igualdad proviene de las Definiciones 1.1.1 y 1.1.5 y del Ejemplo
1.1.3 de *Alpha Calculus*, pp. **1–2 impresas / 10–11 del PDF**.
Los ejemplos que siguen son nuestra elaboración didáctica.

In [2]:
def log_positivo(valor):
    """Coordenada de una cantidad adimensional, positiva y finita."""
    if not math.isfinite(valor) or valor <= 0:
        raise ValueError("La cantidad debe ser positiva y finita.")
    return math.log(valor)


def producto_desde_log(total):
    """Recupera un producto positivo representable en float."""
    if not math.isfinite(total):
        raise ValueError("La suma logarítmica debe ser finita.")
    producto = math.exp(total)
    if producto == 0.0:
        raise ArithmeticError("El producto positivo se redondeó a cero.")
    return producto

In [3]:
def tabla(titulo, cabecera, filas):
    """Imprime una tabla pequeña, con las columnas alineadas."""
    datos = [[str(valor) for valor in fila] for fila in [cabecera, *filas]]
    anchos = [max(map(len, columna)) for columna in zip(*datos)]
    print(titulo)
    for indice, fila in enumerate(datos):
        print(" | ".join(
                valor.ljust(ancho)
                for valor, ancho in zip(fila, anchos)
                ))
        if indice == 0:
            print("-+-".join("-" * ancho for ancho in anchos))


def exportar_tabla(nombre, cabecera, filas):
    """Guarda los datos de una tabla como CSV UTF-8."""
    with (RESULTADOS / nombre).open("w", newline="", encoding="utf-8") as archivo:
        escritor = csv.writer(archivo)
        escritor.writerow(cabecera)
        escritor.writerows(filas)

## 2. Aplicación: cambia una etapa de una cadena de ganancias

### Situación y señales de reconocimiento

Una cadena de etapas aplica ganancias adimensionales $g_i>0$.
Su efecto se **multiplica**:

$$G=\prod_{i=1}^n g_i.$$

Queremos reemplazar repetidamente una etapa conocida y consultar el total.
La señal útil es la combinación de **composición multiplicativa** y
**cambios locales**. No estamos sumando beneficios monetarios.

**Supuestos:** factores positivos y finitos; índice conocido; composición
multiplicativa válida para el problema. Si los factores dependen del estado
de otras etapas, hay que modelar esa dependencia antes de usar esta receta.

### Paso 1. Hacer visibles las contribuciones

Para $g=[2,3,5]$, guardamos $b_i=\log g_i$ y
$S=\sum_i b_i$. El producto es $e^S=30$.

In [4]:
ganancias = [2.0, 3.0, 5.0]
coordenadas = [log_positivo(valor) for valor in ganancias]
total_log = math.fsum(coordenadas)

filas_etapas = [
    [indice + 1, ganancia, f"{coordenada:.9f}"]
    for indice, (ganancia, coordenada)
    in enumerate(zip(ganancias, coordenadas))
    ]
tabla("Contribución por etapa", ["Etapa", "g", "log(g)"], filas_etapas)
print(f"S = {total_log:.9f}; exp(S) = {math.exp(total_log):.9f}")

assert math.isclose(math.exp(total_log), 30.0, rel_tol=1e-12)
exportar_tabla("ganancias-etapas.csv", ["etapa", "g", "log_g"], filas_etapas)

Contribución por etapa
Etapa | g   | log(g)     
------+-----+------------
1     | 2.0 | 0.693147181
2     | 3.0 | 1.098612289
3     | 5.0 | 1.609437912
S = 3.401197382; exp(S) = 30.000000000


### Paso 2. Sustituir $3\to4$

Retiramos el logaritmo anterior y añadimos el nuevo:

$$S'=S-\log3+\log4=\log40.$$

El incremento $\Delta S=\log(4/3)$ explica el cambio relativo:
$G'/G=4/3$. Comparamos tres rutas para el mismo resultado.

In [5]:
indice = 1
nueva_ganancia = 4.0
ganancias_nuevas = ganancias.copy()
ganancias_nuevas[indice] = nueva_ganancia

total_nuevo = math.fsum(
        [
            total_log,
            -coordenadas[indice],
            log_positivo(nueva_ganancia),
            ]
        )
via_log = producto_desde_log(total_nuevo)
via_cociente = math.prod(ganancias) / ganancias[indice] * nueva_ganancia
via_recalculo = math.prod(ganancias_nuevas)

tabla(
        "Tres rutas, una sustitución",
        ["Ruta", "Nuevo producto"],
        [
            ["Recalcular", f"{via_recalculo:.12g}"],
            ["Producto/cociente", f"{via_cociente:.12g}"],
            ["Suma de logaritmos", f"{via_log:.12g}"],
            ],
        )
assert math.isclose(via_log, 40.0, rel_tol=1e-12)
assert via_cociente == via_recalculo == 40.0

Tres rutas, una sustitución
Ruta               | Nuevo producto
-------------------+---------------
Recalcular         | 40            
Producto/cociente  | 40            
Suma de logaritmos | 40            


### Paso 3. Contar el trabajo pertinente

Con $n$ etapas y $k$ sustituciones por índice conocido, cada una seguida
de una consulta del total, usamos el modelo de precisión fija y operaciones
elementales
de coste unitario. Preparar el estado cuesta $O(n)$ en las tres rutas.

| Ruta | Trabajo por sustitución y consulta | Total |
|---|---|---|
| Recalcular producto | $O(n)$ | $O(n+kn)$ |
| Producto/cociente mantenido | $O(1)$ | $O(n+k)$ |
| Suma de logaritmos mantenida | $O(1)$ | $O(n+k)$ |

La ruta logarítmica necesita un logaritmo nuevo, una suma de tres términos
y una exponencial si pedimos el producto. `fsum` no equivale a una sola
suma de máquina. Guardar los factores o coordenadas usa $O(n)$ de memoria.

**Qué aporta aquí Alpha:** una guía coherente para elegir representación
y explicar contribuciones. La mejora asintótica procede de conservar
estado; producto/cociente tiene el mismo orden. Un método convencional
que conserve logaritmos es este mismo algoritmo. No medimos tiempos.

### Paso 4. Comprobar el workload, sin convertirlo en garantía universal

Repetimos 256 sustituciones sobre 16 ganancias, con semilla fija.
Son **potencias de dos**: el baseline producto/cociente coincide
exactamente con la referencia racional en este workload. Esa exactitud
no se extiende a factores float arbitrarios.

In [6]:
azar = random.Random(975)
factores = [2.0 ** azar.randint(-3, 3) for _ in range(16)]
logs = [math.log(valor) for valor in factores]
suma = math.fsum(logs)
producto_cociente = math.prod(factores)
error_maximo = 0.0

for _ in range(256):
    posicion = azar.randrange(len(factores))
    nuevo = 2.0 ** azar.randint(-3, 3)
    nuevo_log = math.log(nuevo)
    suma = math.fsum([suma, -logs[posicion], nuevo_log])
    producto_cociente = producto_cociente / factores[posicion] * nuevo
    factores[posicion] = nuevo
    logs[posicion] = nuevo_log
    referencia = float(math.prod(Fraction(valor) for valor in factores))
    calculado = producto_desde_log(suma)
    assert producto_cociente == referencia
    assert math.isclose(calculado, referencia, rel_tol=1e-12)
    error_maximo = max(error_maximo, abs(calculado / referencia - 1))

print("Sustituciones comprobadas: 256 / 256")
print(f"Error relativo máximo de la ruta log: {error_maximo:.3e}")
print("Producto/cociente: exacto en este workload de potencias de dos.")

Sustituciones comprobadas: 256 / 256
Error relativo máximo de la ruta log: 2.887e-15
Producto/cociente: exacto en este workload de potencias de dos.


### Paso 5. Conservar rango: logaritmo representable, producto no siempre

El producto $10^{200}10^{200}=10^{400}$ no cabe en el `float`
habitual, aunque su logaritmo $400\log10$ sí. Para $10^{-400}$ ocurre
subdesbordamiento: el cero calculado no es el producto real.

En una cadena $[10^{200},10^{200},10^{-200},10^{-200}]$, el resultado
real es 1. El producto secuencial puede desbordar antes de compensarse.
Reordenar los factores es un baseline convencional útil en este ejemplo.

In [7]:
def describir_retorno(suma):
    try:
        return f"{producto_desde_log(suma):.9g}"
    except OverflowError:
        return "Desbordamiento"
    except ArithmeticError:
        return "Subdesbordamiento a cero"


filas_rango = []
for etiqueta, valores in [
        ("Grande", [1e200, 1e200]),
        ("Pequeño", [1e-200, 1e-200]),
        ("Compensado", [1e200, 1e200, 1e-200, 1e-200]),
        ]:
    suma_rango = math.fsum(math.log(valor) for valor in valores)
    filas_rango.append(
            [
                etiqueta,
                math.prod(valores),
                f"{suma_rango:.6f}",
                describir_retorno(suma_rango),
                ]
            )

tabla(
        "Rango de representación",
        ["Caso", "Producto directo", "S", "exp(S)"],
        filas_rango,
        )
exportar_tabla(
        "ganancias-rango.csv",
        ["caso", "directo", "log_total", "retorno"],
        filas_rango,
        )
print("Producto con factores reordenados:", math.prod([1e200, 1e-200] * 2))
assert describir_retorno(0.0) == "1"

Rango de representación
Caso       | Producto directo | S           | exp(S)                  
-----------+------------------+-------------+-------------------------
Grande     | inf              | 921.034037  | Desbordamiento          
Pequeño    | 0.0              | -921.034037 | Subdesbordamiento a cero
Compensado | inf              | 0.000000    | 1                       
Producto con factores reordenados: 1.0


### Límite: `fsum` no recupera información ya perdida

Partimos de coordenadas $[700,-700,\log(1+\delta)]$, con
$1+\delta=1.00000000000001$. Sustituimos las dos primeras ganancias por
1. En reales queda la pequeña tercera contribución. Un acumulador float
puede perderla durante las actualizaciones, incluso usando `fsum`.

Recalcular desde los componentes recupera la contribución en este ejemplo;
cuesta $O(n)$ y debe incluirse si se adopta una política de reconstrucción.
Ni el contraejemplo ni la reparación son una cota de error general.

In [8]:
factor_pequeno = 1.00000000000001
termino_pequeno = math.log(factor_pequeno)
acumulado = math.fsum([700.0, -700.0, termino_pequeno])
acumulado = math.fsum([acumulado, -700.0, 0.0])
acumulado = math.fsum([acumulado, 700.0, 0.0])
reconstruido = math.fsum([0.0, 0.0, termino_pequeno])

tabla(
        "Cancelación histórica reproducida",
        ["Ruta", "Producto"],
        [
            ["Esperado", f"{factor_pequeno:.17g}"],
            ["Acumulador incremental", f"{math.exp(acumulado):.17g}"],
            ["Reconstrucción", f"{math.exp(reconstruido):.17g}"],
            ],
        )
assert acumulado == 0.0
assert math.exp(acumulado) != factor_pequeno
assert math.exp(reconstruido) == factor_pequeno

Cancelación histórica reproducida
Ruta                   | Producto        
-----------------------+-----------------
Esperado               | 1.00000000000001
Acumulador incremental | 1               
Reconstrucción         | 1.00000000000001


### Ejercicio

La cadena es $[0.5,2,8]$. Cambia la segunda ganancia a 3.
Antes de ejecutar: ¿cuánto vale el nuevo producto? ¿Qué contribución
logarítmica retiras y cuál añades? ¿Por qué una atenuación no exige
usar un logaritmo complejo?

La siguiente celda deja la solución visible para contrastarla después.

In [9]:
ejercicio = [0.5, 2.0, 8.0]
respuesta_log = math.fsum(
        [
            math.fsum(math.log(valor) for valor in ejercicio),
            -math.log(2.0),
            math.log(3.0),
            ]
        )
print("Respuesta:", math.exp(respuesta_log))
assert math.isclose(math.exp(respuesta_log), 12.0, rel_tol=1e-12)

Respuesta: 12.0


## 3. Cuándo aporta valor esta representación

La suma de logaritmos hace visibles las contribuciones multiplicativas
y permite conservar el log-total aunque el producto no quepa en `float`.
Mantener el estado evita recomputar toda la cadena tras cada cambio.

Producto/cociente obtiene el mismo orden de coste. Un método convencional
que conserva logaritmos es este mismo algoritmo. La mejora asintótica
procede del estado mantenido; no se reivindica novedad ni ventaja exclusiva
Alpha. Para elegir una implementación, fija primero rango, precisión y
frecuencia de reconstrucción: la suma precisa de tres operandos no recupera
información ya perdida en un total anterior.

**Extensión propuesta:** estudiar un workload público pertinente, una
política de reconstrucción y requisitos de precisión antes de medir tiempos
o convertir la demostración en una API de producción.

## Fuentes y alcance

- Georgiev y Aydin, *Alpha Calculus*, 1.ª ed., 2026,
  [DOI 10.1201/9781003750734](https://doi.org/10.1201/9781003750734).
  Pasajes cotejados: pp. **1–2 impresas / 10–11 del PDF**, Definiciones
  1.1.1 y 1.1.5 y Ejemplo 1.1.3: generador exponencial y suma transportada.
- [Python: `math`](https://docs.python.org/3.14/library/math.html),
  apartados `log`, `exp`, `fsum`, `prod` e `isclose`: operaciones y límites
  de implementación. El rango y la cancelación se comprueban aquí.

La aplicación, los ejemplos y el código son elaboración didáctica de este
laboratorio, apoyada en una demostración local previa. No se atribuyen como
caso del libro ni como algoritmo científicamente nuevo. Las 256
sustituciones usan potencias de dos: no prueban cotas para floats arbitrarios.
No se distribuye el libro ni se certifica la totalidad de sus resultados.

In [10]:
resumen = {
    "estado": "passed",
    "aplicacion": "cadenas_de_ganancias",
    "python": platform.python_version(),
    "sustituciones": 256,
    "factores": 16,
    "semilla": 975,
    "workload": "Potencias de dos, exponentes enteros entre -3 y 3.",
    "error_relativo_maximo_log": error_maximo,
    "ejemplo_30_a_40": via_log,
    "cancelacion_incremental": math.exp(acumulado),
    "cancelacion_reconstruida": math.exp(reconstruido),
    }
(RESULTADOS / "comprobaciones.json").write_text(
        json.dumps(resumen, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
        )
print("Comprobaciones aprobadas. CSV y resumen en:", RESULTADOS)

Comprobaciones aprobadas. CSV y resumen en: resultados/cadenas_de_ganancias
